# Duolingo Growth & Marketing Operations Analytics

Executed QA and business diagnostics. Analysis as of **March 2026**; corporate scope **Q1 2023–Q4 2025**, website **February 2026**, social **March 2026**. Canonical public data are preserved. DAU/MAU is a ratio of reported average counts, not cohort retention or average individual use.

In [1]:
from pathlib import Path
import sys
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
assert (ROOT/'src/build_project.py').is_file()
sys.path.insert(0, str(ROOT/'src'))
import build_project as build
q, web, social = build.load_inputs(ROOT)
print(f'Validated exact scope: {len(q)} quarters, {len(web)} website rows, {len(social)} social rows.')
q[['quarter','mau_m','dau_m','paid_subscribers_m','revenue_usd_m']]

Validated exact scope: 12 quarters, 3 website rows, 9 social rows.


,quarter,mau_m,dau_m,paid_subscribers_m,revenue_usd_m
0,2023-Q1,72.6,20.3,4.8,115.661
1,2023-Q2,74.1,21.4,5.2,126.839
2,2023-Q3,83.1,24.2,5.8,137.624
3,2023-Q4,88.4,26.9,6.6,150.985
4,2024-Q1,97.6,31.4,7.4,167.553
5,2024-Q2,103.6,34.1,8.0,178.327
6,2024-Q3,113.1,37.2,8.6,192.594
7,2024-Q4,116.7,40.5,9.5,209.600
8,2025-Q1,130.2,46.6,10.3,230.700
9,2025-Q2,128.3,47.7,10.9,252.300


## Independent calculation and output reconciliation

All **16 derived fields** are checked by quarter key, including every missing-value mask. Pandas calculations are reconciled with separate Decimal arithmetic and independently executed SQL formulas. Raw values and all **156** long-format rows are also checked. Growth rates are fractions in CSVs; multiplying by 100 gives percent. Differences of growth rates multiplied by 100 give percentage points.

In [2]:
processed, decimal_diffs, sql_diffs = build.check_processed(q, web, social, ROOT)
assert len(decimal_diffs) == len(sql_diffs) == 16
assert max(decimal_diffs.values()) < 1e-10
assert max(sql_diffs.values()) < 1e-10
pd.DataFrame({'Decimal max absolute difference': decimal_diffs, 'SQL max absolute difference': sql_diffs})

,Decimal max absolute difference,SQL max absolute difference
dau_mau_ratio,7.658402e-17,5.551115e-17
mau_qoq,1.737705e-16,8.326673e-17
mau_yoy,1.859416e-16,5.551115e-17
mau_index,5.096419e-14,2.842171e-14
dau_qoq,2.009852e-16,9.714451e-17
dau_yoy,2.551402e-16,5.551115e-17
dau_index,3.128079e-14,2.842171e-14
paid_subscribers_qoq,2.043478e-16,9.020562e-17
paid_subscribers_yoy,1.615385e-16,5.551115e-17
paid_subscribers_index,3.666667e-14,2.842171e-14


SQL execution uses an in-memory SQLite compatibility adaptation of the actual maintained MySQL scripts: database/use commands are omitted, view replacement becomes view creation, and DOUBLE casts become REAL. Seeds, every validation query, views, all 16 derived fields and latest KPI query pass assertions. **Native MySQL precision, coercion, DDL and server execution are not certified.** Competitor numbers remain inherited historical snapshots; original captures are unavailable, so their provenance is unverified.

In [3]:
start, end = processed.iloc[0], processed.iloc[-1]
summary = pd.DataFrame([
    ['MAU',start.mau_m,end.mau_m,'million users',(end.mau_m/start.mau_m-1)*100,'percent growth'],
    ['DAU',start.dau_m,end.dau_m,'million users',(end.dau_m/start.dau_m-1)*100,'percent growth'],
    ['Paid subscribers',start.paid_subscribers_m,end.paid_subscribers_m,'million accounts',(end.paid_subscribers_m/start.paid_subscribers_m-1)*100,'percent growth'],
    ['Quarterly revenue',start.revenue_usd_m,end.revenue_usd_m,'USD millions',(end.revenue_usd_m/start.revenue_usd_m-1)*100,'percent growth'],
    ['DAU/MAU',start.dau_mau_ratio*100,end.dau_mau_ratio*100,'percent',(end.dau_mau_ratio-start.dau_mau_ratio)*100,'percentage-point increase']
], columns=['Metric','Q1 2023','Q4 2025','Level unit','Change','Change unit'])
summary.round(3)

,Metric,Q1 2023,Q4 2025,Level unit,Change,Change unit
0,MAU,72.600,133.100,million users,83.333,percent growth
1,DAU,20.300,52.700,million users,159.606,percent growth
2,Paid subscribers,4.800,12.200,million accounts,154.167,percent growth
3,Quarterly revenue,115.661,282.900,USD millions,144.594,percent growth
4,DAU/MAU,27.961,39.594,percent,11.633,percentage-point increase


## Growth diagnostics and operational interpretation

Daily activity, paid accounts and revenue expanded faster than MAU across the fixed window. At Q4 2025 their YoY growth gaps versus MAU are **16.1, 14.4 and 20.9 pp**. These descriptive divergences support reviews of engagement and monetisation; they do not establish conversion, pricing effectiveness, retention or commercial efficiency. An authorised internal next step would audit existing measurement and investigate drivers with free-experience guardrails.

In [4]:
fields = ['quarter','mau_yoy','dau_yoy','paid_subscribers_yoy','revenue_yoy','dau_yoy_minus_mau_yoy','paid_subscribers_yoy_minus_mau_yoy','revenue_yoy_minus_mau_yoy']
diagnostics = processed.loc[processed.mau_yoy.notna(), fields].copy()
diagnostics.iloc[:,1:] = diagnostics.iloc[:,1:]*100
diagnostics.columns = ['Quarter','MAU YoY (%)','DAU YoY (%)','Paid YoY (%)','Revenue YoY (%)','DAU minus MAU (pp)','Paid minus MAU (pp)','Revenue minus MAU (pp)']
assert diagnostics.iloc[-1,1:].astype(float).round(1).tolist() == [14.1,30.1,28.4,35.0,16.1,14.4,20.9]
# Use explicit half-up display rounding so 36.25% displays as 36.3%, consistent with SQL.
from decimal import Decimal, ROUND_HALF_UP
for column in diagnostics.columns[1:]:
    diagnostics[column] = diagnostics[column].map(lambda value: float(Decimal(str(value)).quantize(Decimal('0.1'), rounding=ROUND_HALF_UP)))
diagnostics

,Quarter,MAU YoY (%),DAU YoY (%),Paid YoY (%),Revenue YoY (%),DAU minus MAU (pp),Paid minus MAU (pp),Revenue minus MAU (pp)
4,2024-Q1,34.4,54.7,54.2,44.9,20.2,19.7,10.4
5,2024-Q2,39.8,59.3,53.8,40.6,19.5,14.0,0.8
6,2024-Q3,36.1,53.7,48.3,39.9,17.6,12.2,3.8
7,2024-Q4,32.0,50.6,43.9,38.8,18.5,11.9,6.8
8,2025-Q1,33.4,48.4,39.2,37.7,15.0,5.8,4.3
9,2025-Q2,23.8,39.9,36.3,41.5,16.0,12.4,17.6
10,2025-Q3,19.6,35.8,33.7,41.1,16.1,14.1,21.4
11,2025-Q4,14.1,30.1,28.4,35.0,16.1,14.4,20.9
